# 🔬 필터 수 실험: Conv2D(32) vs Conv2D(64)
## 첫 번째 Conv 레이어의 필터 수를 바꿨을 때 파라미터 수·성능 비교

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.callbacks import EarlyStopping

---
## 1. 데이터 로드 (01 노트북과 동일)

In [ ]:
IMG_SIZE = 64
DATA_DIR = Path("data")


def load_images(folder, label):
    images = []
    valid_ext = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    for img_path in sorted(folder.iterdir()):
        if img_path.suffix.lower() not in valid_ext:
            continue
        try:
            img = Image.open(img_path).convert("L")
            img = img.resize((IMG_SIZE, IMG_SIZE))
            images.append(np.array(img))
        except Exception:
            pass
    return np.array(images), np.full(len(images), label)


# 데이터 로드
my_train, my_train_y = load_images(DATA_DIR / "train" / "my_cat", 1)
other_train, other_train_y = load_images(DATA_DIR / "train" / "other_cats", 0)
my_val, my_val_y = load_images(DATA_DIR / "val" / "my_cat", 1)
other_val, other_val_y = load_images(DATA_DIR / "val" / "other_cats", 0)

train_images = np.concatenate([my_train, other_train]).astype("float32") / 255.0
train_labels = np.concatenate([my_train_y, other_train_y])
val_images = np.concatenate([my_val, other_val]).astype("float32") / 255.0
val_labels = np.concatenate([my_val_y, other_val_y])

train_images = train_images.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
val_images = val_images.reshape(-1, IMG_SIZE, IMG_SIZE, 1)

print(f"학습: {train_images.shape}, 검증: {val_images.shape}")

---
## 2. 모델 A: Conv2D(**32**) → Conv2D(64)

In [ ]:
def build_model(first_filters: int):
    """첫 번째 Conv 필터 수만 다른 동일 구조 모델 생성"""
    return keras.Sequential([
        # 첫 번째 Conv→Pool 블록 (필터 수 변경 지점)
        layers.Conv2D(first_filters, kernel_size=3, activation="relu",
                      padding="same", input_shape=(IMG_SIZE, IMG_SIZE, 1)),
        layers.MaxPooling2D(pool_size=2),

        # 두 번째 Conv→Pool 블록
        layers.Conv2D(first_filters * 2, kernel_size=3, activation="relu",
                      padding="same"),
        layers.MaxPooling2D(pool_size=2),

        # 분류기
        layers.Flatten(),
        layers.Dense(100, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid"),
    ])


# 모델 A: 첫 번째 Conv = 32
model_32 = build_model(first_filters=32)
print("=" * 60)
print("모델 A: Conv2D(32) → Conv2D(64)")
print("=" * 60)
model_32.summary()

---
## 3. 모델 B: Conv2D(**64**) → Conv2D(128)

In [ ]:
model_64 = build_model(first_filters=64)
print("=" * 60)
print("모델 B: Conv2D(64) → Conv2D(128)")
print("=" * 60)
model_64.summary()

---
## 4. 파라미터 수 직접 계산 & 비교

In [ ]:
print("=" * 65)
print(f"{'레이어':<25} {'모델A (32)':<18} {'모델B (64)':<18}")
print("=" * 65)

for layer_a, layer_b in zip(model_32.layers, model_64.layers):
    print(f"{layer_a.name:<25} {layer_a.count_params():<18,} {layer_b.count_params():<18,}")

print("-" * 65)
print(f"{'총 파라미터':<25} {model_32.count_params():<18,} {model_64.count_params():<18,}")
print(f"{'차이':<25} {model_64.count_params() - model_32.count_params():>+17,}")

### 파라미터 수 수동 계산

**모델 A (첫 Conv = 32)**
| 레이어 | 계산 | 파라미터 |
|--------|------|----------|
| Conv2D(32, 3) | (3×3×1+1)×32 | 320 |
| Conv2D(64, 3) | (3×3×32+1)×64 | 18,496 |
| Dense(100) | 16×16×64×100+100 | 1,638,500 |
| Dense(1) | 100×1+1 | 101 |
| **총** | | **1,657,417** |

**모델 B (첫 Conv = 64)**
| 레이어 | 계산 | 파라미터 |
|--------|------|----------|
| Conv2D(64, 3) | (3×3×1+1)×64 | 640 |
| Conv2D(128, 3) | (3×3×64+1)×128 | 73,856 |
| Dense(100) | 16×16×128×100+100 | 3,276,900 |
| Dense(1) | 100×1+1 | 101 |
| **총** | | **3,351,497** |

→ 필터 수를 **2배**로 늘리면 파라미터는 약 **2배 이상** 증가!

---
## 5. 성능 비교 학습

In [ ]:
def train_model(model, name: str):
    """모델 학습 후 history 반환"""
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
        jit_compile=False,
    )

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=5,
                      restore_best_weights=True, verbose=0),
    ]

    print(f"\n🏋️ {name} 학습 시작...")
    history = model.fit(
        train_images, train_labels,
        epochs=30,
        batch_size=16,
        validation_data=(val_images, val_labels),
        callbacks=callbacks,
        verbose=1,
    )
    return history


history_32 = train_model(model_32, "모델A: Conv2D(32)")
history_64 = train_model(model_64, "모델B: Conv2D(64)")

---
## 6. 결과 비교

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 정확도 비교
ax1.plot(history_32.history["val_accuracy"], label="Conv2D(32)", linewidth=2)
ax1.plot(history_64.history["val_accuracy"], label="Conv2D(64)", linewidth=2)
ax1.set_title("검증 정확도 비교", fontsize=13)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Val Accuracy")
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# 손실 비교
ax2.plot(history_32.history["val_loss"], label="Conv2D(32)", linewidth=2)
ax2.plot(history_64.history["val_loss"], label="Conv2D(64)", linewidth=2)
ax2.set_title("검증 손실 비교", fontsize=13)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Val Loss")
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 최종 비교표
best_32 = max(history_32.history["val_accuracy"])
best_64 = max(history_64.history["val_accuracy"])

print("=" * 55)
print(f"{'항목':<20} {'모델A (32)':<16} {'모델B (64)':<16}")
print("=" * 55)
print(f"{'첫 Conv 필터 수':<20} {'32':<16} {'64':<16}")
print(f"{'총 파라미터':<20} {model_32.count_params():<16,} {model_64.count_params():<16,}")
print(f"{'최고 val_accuracy':<20} {best_32:<16.4f} {best_64:<16.4f}")
print(f"{'학습 에포크 수':<20} {len(history_32.history['loss']):<16} {len(history_64.history['loss']):<16}")
print("=" * 55)

if best_64 > best_32:
    print(f"\n→ Conv2D(64)가 {best_64 - best_32:.4f} 더 높은 정확도")
    print(f"  대신 파라미터가 {model_64.count_params() - model_32.count_params():,}개 더 많음")
else:
    print(f"\n→ Conv2D(32)가 더 적은 파라미터로 비슷하거나 더 좋은 성능!")
    print(f"  데이터가 적을 때는 작은 모델이 과적합에 더 강할 수 있음")